# Equip Triage: 장비 이벤트 Transformer
사전학습 모델을 내려받지 않습니다. PyTorch 텐서 연산으로 다중 헤드 어텐션을 직접 구현하고, 합성 검사장비 이벤트의 다음 토큰을 무작위 초기값부터 학습합니다. 아래 셀은 외부 파일 없이 실행되는 CPU 파일럿입니다. Colab에서 **런타임 → 모두 실행**을 누르세요.

미래 이벤트를 가리는 인과 마스크를 씁니다. 완성된 시퀀스에서 가장 큰 점수를 내는 파일럿 지표는 실시간 경보 지연이나 실제 제조 성능을 입증하지 않습니다. 이 노트북은 연구 코드 한 버전을 독립적으로 동결한 것입니다.

## 1. 장비 이벤트와 분할
훈련·임계값 선택·평가용 이벤트 흐름을 분리합니다.

In [ ]:
"""AI-authored reference: synthetic causal event prediction, no pretrained weights.

For target event t, input is [BOS, events[:t]]. Attention cannot see later
positions; the current target is never an input at its prediction position.
Only normal TRAIN scenarios fit models. VALIDATION picks anomaly thresholds.
TEST scenarios are held out by template, not randomly split windows.
"""
import argparse
from collections import Counter, defaultdict
import csv
import json
import math
from pathlib import Path
import random
import time

import torch
from torch import nn
import torch.nn.functional as F

VOCAB = ('BOS', 'START', 'CHECK', 'LOAD', 'MEASURE', 'RETEST', 'UNLOAD', 'END', 'WAIT')
TOKEN = {name: i for i, name in enumerate(VOCAB)}
# Different complete templates in each split. No window crosses a scenario.
TEMPLATES = {
    'train': [('START', 'CHECK', 'LOAD', 'MEASURE', 'UNLOAD', 'END'),
              ('START', 'LOAD', 'CHECK', 'MEASURE', 'RETEST', 'MEASURE', 'UNLOAD', 'END')],
    'validation': [('START', 'CHECK', 'LOAD', 'MEASURE', 'RETEST', 'MEASURE', 'UNLOAD', 'END')],
    'test': [('START', 'LOAD', 'CHECK', 'MEASURE', 'UNLOAD', 'END')],
}


def scenarios(seed=42):
    rng = random.Random(seed)
    rows = []
    for split, templates in TEMPLATES.items():
        for template_id, template in enumerate(templates):
            # Each split also has a disjoint number of consecutive measurements.
            counts = {'train': range(1, 7), 'validation': range(7, 10), 'test': range(10, 13)}[split]
            for repeats in counts:
                normal = []
                for event in template:
                    normal.extend([event] * (repeats if event == 'MEASURE' else 1))
                cases = [('normal', normal)]
                if split != 'train':
                    first = normal.index('MEASURE')
                    missing = normal[:first] + normal[first + repeats:]
                    corrupt = list(normal)
                    corrupt[first] = 'END'
                    novel = list(normal)
                    novel[first] = 'WAIT'
                    cases += [('missing_measure', missing), ('early_end', corrupt), ('unexpected_wait', novel)]
                for label, events in cases:
                    rows.append({'split': split, 'scenario': f'{split}-template{template_id}-repeat{repeats}-{label}',
                                 'template': f'{split}-template{template_id}', 'label': label, 'events': events})
    rng.shuffle(rows)
    # A full normal sequence is not duplicated across splits.
    signatures = defaultdict(set)
    for row in rows:
        signature = tuple(row['events'])
        signatures[signature].add(row['split'])
    assert all(len(splits) == 1 for splits in signatures.values())
    return rows

## 2. 모델 직접 구현
Q/K/V 투영, 인과 마스크, 다중 헤드 어텐션, 위치 임베딩, 잔차 블록을 확인하세요.

In [ ]:
class Attention(nn.Module):
    def __init__(self, width, heads):
        super().__init__()
        assert width % heads == 0
        self.heads, self.depth = heads, width // heads
        self.qkv = nn.Linear(width, 3 * width)
        self.out = nn.Linear(width, width)

    def forward(self, x):
        batch, length, width = x.shape
        q, k, v = self.qkv(x).chunk(3, dim=-1)
        q, k, v = [z.reshape(batch, length, self.heads, self.depth).transpose(1, 2) for z in (q, k, v)]
        scores = (q @ k.transpose(-2, -1)) / math.sqrt(self.depth)
        future = torch.ones(length, length, dtype=torch.bool, device=x.device).triu(1)
        scores = scores.masked_fill(future, float('-inf'))
        weights = scores.softmax(dim=-1)
        result = (weights @ v).transpose(1, 2).contiguous().reshape(batch, length, width)
        return self.out(result)


class TinyTransformer(nn.Module):
    def __init__(self, width=32, heads=4, max_length=96):
        super().__init__()
        self.embedding = nn.Embedding(len(VOCAB), width)
        position = torch.arange(max_length).unsqueeze(1)
        scale = torch.exp(torch.arange(0, width, 2) * (-math.log(10000.0) / width))
        pe = torch.zeros(max_length, width)
        pe[:, 0::2], pe[:, 1::2] = torch.sin(position * scale), torch.cos(position * scale)
        self.register_buffer('position', pe)
        self.norm1, self.norm2 = nn.LayerNorm(width), nn.LayerNorm(width)
        self.attention = Attention(width, heads)
        self.ff = nn.Sequential(nn.Linear(width, 2 * width), nn.GELU(), nn.Linear(2 * width, width))
        self.output = nn.Linear(width, len(VOCAB))

    def forward(self, ids):
        x = self.embedding(ids) + self.position[:ids.shape[1]]
        x = x + self.attention(self.norm1(x))  # residual 1
        x = x + self.ff(self.norm2(x))        # residual 2
        return self.output(x)

## 3. 기준선과 평가
같은 입력에서 다음 이벤트 확률을 비교합니다. 이상 임계값은 검증셋에서만 선택합니다.

In [ ]:
def batch(rows):
    length = max(len(row['events']) for row in rows)
    inputs = torch.zeros(len(rows), length, dtype=torch.long)
    targets = torch.full_like(inputs, -100)  # ignored padding in masked loss
    for i, row in enumerate(rows):
        ids = [TOKEN[e] for e in row['events']]
        inputs[i, :len(ids)] = torch.tensor([TOKEN['BOS']] + ids[:-1])
        targets[i, :len(ids)] = torch.tensor(ids)
    return inputs, targets


class NGram:
    def __init__(self, rows, order=3):
        self.order = order
        self.counts = defaultdict(Counter)
        for row in rows:
            history = [TOKEN['BOS']]
            for event in row['events']:
                token = TOKEN[event]
                for size in range(order):
                    context = tuple(history[-size:]) if size else ()
                    self.counts[context][token] += 1
                history.append(token)

    def losses(self, events):
        history, values = [TOKEN['BOS']], []
        for event in events:
            token = TOKEN[event]
            for size in range(min(self.order - 1, len(history)), -1, -1):
                context = tuple(history[-size:]) if size else ()
                counts = self.counts.get(context)
                if counts:
                    break
            # Add-one smoothing, including unseen event IDs.
            values.append(-math.log((counts[token] + 1) / (sum(counts.values()) + len(VOCAB))))
            history.append(token)
        return values


def transformer_losses(model, events):
    x, y = batch([{'events': events}])
    with torch.no_grad():
        return F.cross_entropy(model(x).reshape(-1, len(VOCAB)), y.reshape(-1), reduction='none').tolist()


def metrics(rows, score, threshold):
    tp = fp = tn = fn = 0
    details = []
    for row in rows:
        value = score(row['events'])
        predicted, actual = value > threshold, row['label'] != 'normal'
        tp += predicted and actual
        fp += predicted and not actual
        tn += not predicted and not actual
        fn += not predicted and actual
        details.append({'scenario': row['scenario'], 'label': row['label'], 'score': round(value, 6), 'flagged': predicted})
    return {'tp': tp, 'fp': fp, 'tn': tn, 'fn': fn,
            'anomaly_f1': round(2 * tp / max(1, 2 * tp + fp + fn), 6), 'scenarios': details}


def evaluate(rows, losses):
    # Maximum observed event surprise; online score at t needs no event after t.
    score = lambda events: max(losses(events))
    validation = [r for r in rows if r['split'] == 'validation']
    test = [r for r in rows if r['split'] == 'test']
    candidates = sorted({score(r['events']) for r in validation})
    threshold = max(candidates, key=lambda t: (metrics(validation, score, t)['anomaly_f1'], t))
    return {'validation_threshold': threshold, 'validation': metrics(validation, score, threshold),
            'test': metrics(test, score, threshold)}

## 4. 처음부터 학습
결과가 동률이면 Transformer가 더 낫다고 쓰지 않습니다.

In [ ]:
def run(seed=42, epochs=120):
    torch.set_num_threads(1)
    torch.manual_seed(seed)
    rows = scenarios(seed)
    train = [r for r in rows if r['split'] == 'train']
    model = TinyTransformer()  # Entirely random initialization, no download path.
    optimizer = torch.optim.AdamW(model.parameters(), lr=0.003)
    x, target = batch(train)
    started = time.perf_counter()
    for _ in range(epochs):
        model.train()
        optimizer.zero_grad()
        logits = model(x)
        # Causal prediction + ignored padding. No bidirectional masked-token objective.
        loss = F.cross_entropy(logits.reshape(-1, len(VOCAB)), target.reshape(-1), ignore_index=-100)
        loss.backward()
        optimizer.step()
    elapsed = time.perf_counter() - started
    model.eval()
    baseline = NGram(train)
    result = {'scope': 'synthetic template-disjoint pilot; not production evidence',
              'seed': seed, 'epochs': epochs, 'torch': torch.__version__, 'device': 'cpu', 'threads': 1,
              'parameters': sum(p.numel() for p in model.parameters()), 'training_seconds': round(elapsed, 4),
              'training_final_loss': round(loss.item(), 6),
              'split_counts': {s: sum(r['split'] == s for r in rows) for s in TEMPLATES},
              'ngram': evaluate(rows, baseline.losses),
              'transformer': evaluate(rows, lambda events: transformer_losses(model, events))}
    return rows, result

In [ ]:
rows, result = run(seed=42, epochs=120)
print("파라미터:", result["parameters"], "| 데이터:", result["split_counts"])
for name in ("ngram", "transformer"):
    print(name, {k: result[name]["test"][k] for k in ("tp", "fp", "tn", "fn", "anomaly_f1")})

## 포트폴리오에 남길 직접 기여
새 검사 시나리오를 본인이 만들고 왜 정상/이상인지 기록하세요. 특히 최근 이벤트는 같고 훨씬 이전 설정만 달라야 정답이 달라지는 사례를 넣어, n-gram과 Transformer의 오류를 나란히 설명할 수 있습니다. 이 노트북이 대신 구현한 코드와 본인이 변경하고 검증한 코드는 커밋에서 분리하세요.